In [41]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, roc_auc_score
from lightgbm import LGBMClassifier


In [42]:
DATA_PATH = "app/data/FINAL_Balanced_Behavioral_Loan_Risk_Dataset(I).csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)


Dataset shape: (3381, 18)


In [43]:
# Drop non-predictive identifier
if "Customer_ID" in df.columns:
    df = df.drop(columns=["Customer_ID"])


In [44]:
df.head()

,Gender,Age_Group,Region,Monthly_Income,Loan_Amount,Interest_Rate,Loan_Purpose,Loan_Status,Avg_Transaction_Freq,Avg_Transaction_Amount,Spending_Score,Behavioral_Anomaly_Index,Credit_Score,Payment_Irregularity,Default_Risk_Score,Transaction_Inconsistency,Inconsistency_Flag
0,Male,<25,Urban,3283.58,19352.47,8.70,Personal,Approved,264.0,1587.46,0.44,0.47,416.0,0.41,0.45,0.19,0
1,Male,41-60,Semi-Urban,2983.59,47585.00,15.84,Home,Approved,29.0,565.50,0.98,0.72,579.0,0.25,0.53,0.57,0
2,Female,25-40,Semi-Urban,6700.04,36867.70,22.46,Personal,Approved,67.0,1296.27,0.81,0.49,474.0,0.86,0.64,0.33,0
3,Female,25-40,Urban,3198.18,30334.27,19.64,Car,Approved,121.0,1747.89,0.74,0.35,594.0,0.70,0.49,0.82,1
4,Female,25-40,Urban,9606.02,8644.91,21.13,Car,Approved,205.0,92.87,0.54,0.68,750.0,0.55,0.63,0.18,0


In [45]:
df = df.dropna(subset=["Loan_Status"])

df["Loan_Status"] = df["Loan_Status"].str.strip().str.lower()

# 1 = BAD (Rejected / Defaulted)
# 0 = GOOD (Approved)
df["target"] = df["Loan_Status"].map({
    "approved": 0,
    "rejected": 1,
    "defaulted": 1
})

print("Target distribution:")
print(df["target"].value_counts())


Target distribution:
target
0    1727
1    1654
Name: count, dtype: int64


In [46]:
df["Income_Loan_Ratio"] = df["Monthly_Income"] / (df["Loan_Amount"] + 1)

df["Behavior_Risk_Mean"] = (
    df["Payment_Irregularity"]
    + df["Behavioral_Anomaly_Index"]
    + df["Transaction_Inconsistency"]
) / 3

df["Transaction_Intensity"] = (
    df["Avg_Transaction_Freq"] * df["Avg_Transaction_Amount"]
)

df["Risk_to_Income"] = df["Credit_Score"] / (df["Monthly_Income"] + 1)


In [47]:
categorical_cols = ["Gender", "Age_Group", "Region", "Loan_Purpose"]

numeric_cols = [
    col for col in df.columns
    if col not in categorical_cols + ["Loan_Status", "target"]
]

X_cat = df[categorical_cols]
X_num = df[numeric_cols]
y = df["target"]


In [48]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False   # IMPORTANT for your sklearn version
)

X_cat_encoded = encoder.fit_transform(X_cat)

encoded_cat_df = pd.DataFrame(
    X_cat_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df.index
)

X = pd.concat([X_num, encoded_cat_df], axis=1)

print("Final feature matrix shape:", X.shape)


Final feature matrix shape: (3381, 31)


In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [50]:
model = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=7,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 1240, number of negative: 1295
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000294 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2899
[LightGBM] [Info] Number of data points in the train set: 2535, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.05
,n_estimators,700
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [51]:
y_pred = model.predict(X_test)
y_prob_bad = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob_bad))


              precision    recall  f1-score   support

           0       0.69      0.50      0.58       432
           1       0.60      0.76      0.67       414

    accuracy                           0.63       846
   macro avg       0.64      0.63      0.62       846
weighted avg       0.64      0.63      0.62       846

ROC AUC: 0.6820372606906423


In [52]:
def test_case(name, data):
    df_test = pd.DataFrame([data])

    # Feature engineering
    df_test["Income_Loan_Ratio"] = df_test["Monthly_Income"] / (df_test["Loan_Amount"] + 1)

    df_test["Behavior_Risk_Mean"] = (
        df_test["Payment_Irregularity"]
        + df_test["Behavioral_Anomaly_Index"]
        + df_test["Transaction_Inconsistency"]
    ) / 3

    df_test["Transaction_Intensity"] = (
        df_test["Avg_Transaction_Freq"] * df_test["Avg_Transaction_Amount"]
    )

    df_test["Risk_to_Income"] = df_test["Credit_Score"] / (df_test["Monthly_Income"] + 1)

    # Encode categoricals
    cat_data = encoder.transform(df_test[categorical_cols])
    cat_df = pd.DataFrame(
        cat_data,
        columns=encoder.get_feature_names_out(categorical_cols)
    )

    num_df = df_test.drop(columns=categorical_cols, errors="ignore")

    X_final = pd.concat(
        [num_df.reset_index(drop=True), cat_df.reset_index(drop=True)],
        axis=1
    )

    X_final = X_final.reindex(columns=X.columns, fill_value=0)

    prob_bad = model.predict_proba(X_final)[0][1]
    decision = "Rejected" if prob_bad >= 0.5 else "Approved"

    print(f"\n{name}")
    print("Probability of rejection:", round(prob_bad, 3))
    print("Decision:", decision)


In [53]:
approve_case = {
    "Gender": "Male",
    "Age_Group": "26-35",
    "Region": "Urban",
    "Loan_Purpose": "Education",
    "Monthly_Income": 65000,
    "Loan_Amount": 200000,
    "Interest_Rate": 10.0,
    "Avg_Transaction_Freq": 50,
    "Avg_Transaction_Amount": 1800,
    "Payment_Irregularity": 0.1,
    "Behavioral_Anomaly_Index": 0.1,
    "Transaction_Inconsistency": 0.1,
    "Credit_Score": 750,
    "Spending_Score": 80
}

reject_case = {
    "Gender": "Male",
    "Age_Group": "18-25",
    "Region": "Rural",
    "Loan_Purpose": "Personal",
    "Monthly_Income": 18000,
    "Loan_Amount": 500000,
    "Interest_Rate": 16.0,
    "Avg_Transaction_Freq": 8,
    "Avg_Transaction_Amount": 500,
    "Payment_Irregularity": 0.9,
    "Behavioral_Anomaly_Index": 0.85,
    "Transaction_Inconsistency": 0.8,
    "Credit_Score": 520,
    "Spending_Score": 30
}

test_case("EXPECTED APPROVAL", approve_case)
test_case("EXPECTED REJECTION", reject_case)



EXPECTED APPROVAL
Probability of rejection: 0.098
Decision: Approved

EXPECTED REJECTION
Probability of rejection: 0.018
Decision: Approved


In [39]:
# Check average predicted probability by true label
probs = model.predict_proba(X_test)[:, 1]

print("Mean prob for target=1 (Approved):", probs[y_test == 1].mean())
print("Mean prob for target=0 (Rejected):", probs[y_test == 0].mean())


Mean prob for target=1 (Approved): 0.5360754528493067
Mean prob for target=0 (Rejected): 0.39337685136539235


In [40]:
feat_imp = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feat_imp.head(10)

Interest_Rate               865
Spending_Score              808
Payment_Irregularity        793
Credit_Score                789
Avg_Transaction_Freq        743
Behavior_Risk_Mean          694
Monthly_Income              686
Avg_Transaction_Amount      683
Loan_Amount                 667
Behavioral_Anomaly_Index    615
dtype: int32